# 모델 학습

config를 읽어 OCSR 모델을 학습합니다. 작업 디렉토리는 이 노트북이 있는 폴더여야 합니다.

## 1. 설정 변수

- `CONFIG_NAME`: `configs/` 폴더의 설정 파일 이름(확장자 제외). 결과는 `outputs/{CONFIG_NAME}/`에 저장됩니다.
- `RESUME`: 중단한 학습을 이어서 하려면 `True`로 바꿉니다.

이 셀만 고치면 됩니다. 아래 셀은 그대로 실행하세요.

In [ ]:
CONFIG_NAME = "config002"
RESUME = False

## 2. 설정 읽기

필요한 모듈을 불러오고, 설정을 읽어 출력합니다.

In [ ]:
import json

import matplotlib.pyplot as plt
import torch

from scripts import config as config_utils
from scripts import datautils, logutils, network, trainer

config = config_utils.load_config(CONFIG_NAME)
print(json.dumps(config, indent=2))

## 3. 환경 확인

패키지 버전과 학습에 쓸 장치를 확인합니다.

In [ ]:
for name, value in logutils.collect_env().items():
    print(f"{name}: {value}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU 메모리: 전체 {total / 1024**3:.1f} GiB, 남은 양 {free / 1024**3:.1f} GiB")

## 4. 데이터 확인

토크나이저의 어휘 크기와 데이터 개수를 확인하고, 증강이 적용된 훈련 이미지를 그려 봅니다. 여기서는 파일을 저장하지 않습니다.

In [ ]:
tokenizer = datautils.build_tokenizer(config)
split = datautils.make_split(config)
train_loader, val_loader = datautils.build_dataloaders(config, tokenizer, split)
print(f"어휘 크기: {tokenizer.vocab_size}")
print(f"훈련 표본: {len(train_loader.dataset)}개, 검증 표본: {len(val_loader.dataset)}개")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    sample = train_loader.dataset[i]
    ax.imshow(datautils.to_display_image(sample["image"], config))
    ax.set_title(sample["image_id"])
    ax.axis("off")
    print(sample["image_id"], sample["smiles"])
    print("  ", tokenizer.tokenize(sample["smiles"]))
plt.show()

## 5. 모델 확인

모델의 구조와 파라미터 수를 확인합니다. 구조만 보는 것이므로 사전학습 가중치는 불러오지 않습니다.

In [ ]:
model = network.build_model(config, tokenizer, load_pretrained=False)
print(model)
print(network.count_parameters(model))

## 6. 학습

학습을 실행합니다. 도중에 멈추려면 노트북의 중단 버튼을 누르세요. 그 시점까지의 상태가 저장되며 `RESUME = True`로 이어서 학습할 수 있습니다.

In [ ]:
result = trainer.fit(CONFIG_NAME, resume=RESUME)

## 7. 결과 요약

최고 검증 Tanimoto와 저장 위치를 확인합니다.

In [ ]:
print(f"최고 검증 Tanimoto: {result['best_tanimoto']} (step {result['best_step']})")
print(f"마지막 step: {result['last_step']}")
print(f"체크포인트: {config_utils.ckpts_dir(CONFIG_NAME)}")
print(f"로그: {config_utils.db_path(CONFIG_NAME)}")